In [1]:
%%capture
!pip install ultralytics

In [7]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import torch
import glob
from PIL import Image
from tqdm import tqdm
import os

# Load a model
model = YOLO("yolo11n.pt")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
os.makedirs('runs/detect/train2', exist_ok=True)

In [ ]:
results = model.train(data='/content/drive/MyDrive/abelhas/data.yaml',
                      epochs=100, imgsz=640
                      )

Ultralytics 8.3.13 🚀 Python-3.10.12 torch-2.4.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=/content/drive/MyDrive/abelhas/data.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=T

train: Scanning /content/drive/MyDrive/abelhas/train/labels.cache... 5640 images, 892 backgrounds, 0 corrupt: 100%|██████████| 5640/5640 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1, 4.0), tile_grid_size=(8, 8))



val: Scanning /content/drive/MyDrive/abelhas/valid/labels.cache... 1604 images, 238 backgrounds, 0 corrupt: 100%|██████████| 1605/1605 [00:00<?, ?it/s]


Plotting labels to runs/detect/train/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to runs/detect/train
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100      2.49G      2.244      3.302      1.303         15        640: 100%|██████████| 353/353 [03:58<00:00,  1.48it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 51/51 [00:26<00:00,  1.94it/s]


... (8 outputs left out)

                   all       1605       1950      0.686      0.606      0.644      0.269

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      6/100      2.48G      2.022      1.457      1.237         14        640: 100%|██████████| 353/353 [03:03<00:00,  1.93it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 51/51 [00:24<00:00,  2.05it/s]


                   all       1605       1950      0.695      0.669       0.68      0.263

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      7/100      2.47G      1.998      1.456      1.227         55        640:  33%|███▎      | 116/353 [01:01<02:30,  1.57it/s]

In [ ]:
def plot_results(results_dir, images_per_row=3, show_titles=True):
    png_files = glob.glob(f'{results_dir}/*.png')

    if len(png_files) == 0:
        print(f"Nenhuma imagem PNG encontrada no diretório: {results_dir}")
        return

    num_images = len(png_files)
    num_rows = (num_images // images_per_row) + (num_images % images_per_row > 0)

    fig, axes = plt.subplots(num_rows, images_per_row,
                             figsize=(images_per_row * 4, num_rows * 4))

    if num_rows == 1:
        axes = [axes]

    for i, png_file in enumerate(png_files):
        img = Image.open(png_file)
        row = i // images_per_row
        col = i % images_per_row

        ax = axes[row][col] if num_rows > 1 else axes[col]
        ax.imshow(img)
        ax.axis('off')

        if show_titles:
            ax.set_title(png_file.split('/')[-1])

    for i in range(num_images, num_rows * images_per_row):
        fig.delaxes(axes.flatten()[i])

    plt.tight_layout()
    plt.show()

In [ ]:
images_valid = "/content/drive/MyDrive/abelhas/valid/images"
tests = images_valid[:10]

for image_path in tqdm(tests):
    results = model(image_path)
    plot_results(results, image_path)